In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>Versões dos pacotes</b></summary>

O código nesta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar essas versões ou mais recentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Além de [visualizar instruções em um Circuit](/guides/visualize-circuits), você pode querer visualizar o agendamento de um Circuit usando o método [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) do Qiskit. Essa visualização pode ajudar você a identificar rapidamente o tempo ocioso nos Qubits, por exemplo. No entanto, esse método não retorna resultados precisos para circuits dinâmicos. Para visualizar o agendamento de circuits dinâmicos, use o método `draw_circuit_schedule_timing`, conforme descrito na seção de [suporte ao Qiskit Runtime](#qr-support).

## Exemplos

Para visualizar um programa de Circuit agendado, você pode chamar essa função com um conjunto de argumentos de controle. A maior parte da aparência da imagem de saída pode ser modificada por uma folha de estilo, mas isso não é obrigatório.

### Desenhar com a folha de estilo padrão

In [1]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

backend = GenericBackendV2(5)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

draw(isa_circuit, target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg" alt="Output of the previous code cell" />

![Saída da célula de código anterior](../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg)

### Desenhar com uma folha de estilo adequada para depuração de programas

In [2]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw, IQXDebugging
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

backend = GenericBackendV2(5)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
draw(isa_circuit, style=IQXDebugging(), target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg" alt="Output of the previous code cell" />

![Saída da célula de código anterior](../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg)

Você pode criar funções personalizadas de gerador ou layout e atualizar uma folha de estilo existente com essas funções personalizadas. Dessa forma, você pode controlar a maior parte da aparência da imagem de saída sem modificar a base de código do visualizador de Circuit agendado. Consulte a referência da API do [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) para mais exemplos.
<span id="qr-support"></span>

## Suporte ao Qiskit Runtime

Embora o visualizador de linha do tempo integrado ao Qiskit seja útil para circuits estáticos, ele pode não refletir com precisão o tempo de execução de [circuits dinâmicos](/guides/classical-feedforward-and-control-flow) devido a operações implícitas, como broadcasting e determinação de branch. Como parte do suporte a circuits dinâmicos, o Qiskit Runtime retorna as informações precisas de tempo do Circuit dentro dos resultados do job quando solicitado.

> **Note:** - Esta é uma função experimental. Ela está em status de versão prévia e, portanto, sujeita a alterações.
> - Esta função se aplica somente a jobs do Qiskit Runtime Sampler.
> - Embora o tempo total do Circuit seja retornado nos metadados de "compilação", esse NÃO é o tempo usado para cobrança (quantum time).

### Habilitar a recuperação de dados de tempo

Para habilitar a recuperação de dados de tempo, defina o sinalizador experimental `scheduler_timing` como `True` ao executar o job do primitivo.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

sampler = SamplerV2(backend)
sampler.options.experimental = {
    "execution": {
        "scheduler_timing": True,
    },
}

sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

### Acessar os dados de tempo do Circuit
Quando solicitados, os dados de tempo do Circuit para cada PUB são retornados nos metadados do resultado do job, em `["compilation"]["scheduler_timing"]["timing"]`. Esse campo contém as informações de tempo brutas. Para exibir as informações de tempo, use a ferramenta de visualização integrada, conforme descrito na seção [Visualizar os tempos](#visualize-timings).

Use o código a seguir para acessar os dados de tempo do Circuit para o primeiro PUB:

In [4]:
job_result = sampler_job.result()
circuit_schedule = job_result[0].metadata["compilation"]["scheduler_timing"]
circuit_schedule_timing = circuit_schedule["timing"]

#### Entender os dados de tempo brutos
Embora a visualização dos dados de tempo do Circuit usando o método `draw_circuit_schedule_timing` seja o caso de uso mais comum, pode ser útil entender a estrutura dos dados de tempo brutos retornados. Isso pode ajudar você, por exemplo, a extrair informações de forma programática.

Os dados de tempo retornados em `["compilation"]["scheduler_timing"]["timing"]` são uma lista de strings. Cada string representa uma única instrução em algum canal e é separada por vírgulas nos seguintes tipos de dados:

- `Branch` - Determina se a instrução está em um fluxo de controle (then / else) ou em um branch principal.
- `Instruction` - O Gate e o Qubit em que operar.
- `Channel` - O canal que está sendo atribuído à instrução. Pode ser um dos seguintes:
      - `Qubit x` - O canal de drive do Qubit _x_.
      - `AWGRx_y` (gerador de forma de onda arbitrária de leitura) - Usado por canais de leitura para comunicar quando os Qubits estão sendo medidos. Os argumentos _x_ e _y_ correspondem ao ID do instrumento de leitura e ao número do Qubit, respectivamente.
- `T0` - O tempo de início da instrução dentro do agendamento completo.
- `Duration` - A duração da instrução, em unidades de _dt_ segundos, onde 1 dt = 1 ciclo de agendamento. Você pode encontrar o valor `dt` de um Backend usando [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- `Pulse` - O tipo de operação de pulso sendo usada.

Exemplo:

In [5]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

# Create a figure from the metadata
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule_timing,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

<span id="visualize-timings"></span>
### Visualizar os tempos
Com `qiskit-ibm-runtime` v0.43.0 ou posterior, você pode visualizar os tempos do Circuit. Para visualizar os tempos, primeiro é necessário converter os metadados do resultado em `fig` usando o [método `draw_circuit_schedule_timing`.](https://github.com/Qiskit/qiskit-ibm-runtime/blob/3d1bf1e1d49e5123841639fce259859c90ce9314/qiskit_ibm_runtime/visualization/draw_circuit_schedule_timings.py#L26) Esse método retorna uma figura `plotly`, que você pode exibir diretamente, salvar em um arquivo ou ambos. Para mais informações sobre os comandos `plotly` a serem usados, consulte [`fig.show()`](https://plotly.com/python-api-reference/generated/plotly.io.show.html) e [`fig.write_image("<path.format>")`.](https://plotly.com/python-api-reference/generated/plotly.io.write_image.html)

In [6]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)

# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)
sampler.options.experimental = {"execution": {"scheduler_timing": True}}

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d5kk3cn853es738e01dg (DONE)


![Passar o cursor sobre a saída mostra informações como início, fim e duração.](../docs/images/guides/visualize-circuit-timing/image_1.svg 'Exemplo de uma figura gerada')
#### Entender a figura gerada
A imagem dos dados de tempo do Circuit gerada por `draw_circuit_schedule_timing` transmite as seguintes informações:

- O eixo X é o tempo em unidades de _dt_ segundos, onde 1 dt = 1 ciclo de agendamento. Você pode encontrar o valor `dt` de um Backend usando [`backend.dt`](https://docs.quantum.ibm.com/api/qiskit/qiskit.providers.BackendV2#dt).
- O eixo Y é o canal (pense nos canais como instrumentos que emitem pulsos).
    - `Receive channel` - O único canal que não é um instrumento por si só. É uma instrução executada em todos os canais que fazem parte de um procedimento de comunicação com o hub naquele momento.
    - `Qubit x` - O canal de drive do Qubit x.
    - `AWGRx_y` (gerador de forma de onda arbitrária de leitura) - Usado por canais de leitura para comunicar quando os Qubits estão sendo medidos. Os argumentos _x_ e _y_ correspondem ao ID do instrumento de leitura e ao número do Qubit, respectivamente.
    - `Hub` - Controla o broadcasting.

Além disso, cada instrução tem o formato *X_Y*, onde *X* é o nome da instrução e *Y* é o tipo de pulso. Um tipo `play` aplica pulsos de controle, e um `capture` registra o estado do Qubit. Você também pode passar o cursor sobre cada instrução para obter mais detalhes. Por exemplo, a figura anterior mostra um pulso de controle para o Gate X aplicado ao Qubit 10 em 1161 dt.
### Exemplo completo
Este exemplo mostra como habilitar a opção, obter as informações de tempo do Circuit a partir dos metadados e exibi-las como uma imagem.

Primeiro, configure o ambiente, defina os circuits e converta-os para circuits ISA, e defina e execute os jobs.

In [7]:
# Get the circuit schedule timing
result[0].metadata["compilation"]["scheduler_timing"]["timing"]

'main,rz_0,Qubit 0,929,0,shift_phase\nmain,sx_0,Qubit 0,929,8,play\nmain,sx_0,Qubit 0,933,0,shift_phase\nmain,rz_0,Qubit 0,937,0,shift_phase\nmain,barrier,Qubit 0,937,0,barrier\nmain,barrier,Qubit 0,937,0,barrier\nmain,barrier,Qubit 1,937,0,barrier\nmain,barrier,Qubit 2,937,0,barrier\nmain,barrier,Qubit 3,937,0,barrier\nmain,barrier,Qubit 4,937,0,barrier\nmain,barrier,Qubit 5,937,0,barrier\nmain,barrier,Qubit 6,937,0,barrier\nmain,barrier,Qubit 7,937,0,barrier\nmain,barrier,Qubit 8,937,0,barrier\nmain,barrier,Qubit 9,937,0,barrier\nmain,barrier,Qubit 10,937,0,barrier\nmain,barrier,Qubit 11,937,0,barrier\nmain,barrier,Qubit 12,937,0,barrier\nmain,barrier,Qubit 13,937,0,barrier\nmain,barrier,Qubit 14,937,0,barrier\nmain,barrier,Qubit 15,937,0,barrier\nmain,barrier,Qubit 16,937,0,barrier\nmain,barrier,Qubit 17,937,0,barrier\nmain,barrier,Qubit 18,937,0,barrier\nmain,barrier,Qubit 19,937,0,barrier\nmain,barrier,Qubit 20,937,0,barrier\nmain,barrier,Qubit 21,937,0,barrier\nmain,barrier,Qubit

Finally, you can visualize and save the timing:

In [8]:
from qiskit_ibm_runtime.visualization import draw_circuit_schedule_timing

circuit_schedule = result[0].metadata["compilation"]["scheduler_timing"][
    "timing"
]
fig = draw_circuit_schedule_timing(
    circuit_schedule=circuit_schedule,
    included_channels=None,
    filter_readout_channels=False,
    filter_barriers=False,
    width=1000,
)

# Uncomment the following line to display the figure
# fig.show(renderer="notebook")

# Save to a file
# fig.write_html("scheduler_timing.html")

Em seguida, obtenha o agendamento de tempo do Circuit: